# StockVision: train the detection model (Google Colab)

Pulls a dataset from Roboflow, trains YOLOv8n, and downloads `best.pt` plus a few sample images.

**Before running:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all.

You need a free Roboflow account for the API key (Roboflow > Settings > API Keys).
Either paste it when asked, or save it once as a Colab secret named `ROBOFLOW_API_KEY` (key icon in the left bar).

## 1. Parameters: the only cell you edit

In [ ]:
# Dataset: taken from the Roboflow URL  universe.roboflow.com/<WORKSPACE>/<PROJECT>
# Our own project (auto-labelled in Roboflow; v2 = 23 images with per-tube tiles, 17 Nov 2026).
ROBOFLOW_WORKSPACE = "khalids-workspace-4w39o"
ROBOFLOW_PROJECT = "stockvisiondraft"
ROBOFLOW_VERSION = 2          # the number on the Versions page

# Classes the pilot counts. Must be spelled exactly like the Roboflow class names
# and like the keys of the `classes` block in config.yaml.
CLASSES = ["cement-bag", "square-tube", "round-pipe"]

EPOCHS = 50                   # 50 is fine for a first pass; 100 for the final model
IMAGE_SIZE = 640
BATCH = 16
BASE_MODEL = "yolov8n.pt"     # nano: small enough for CPU inference on a laptop

## 2. Install and check the GPU

In [ ]:
!pip -q install ultralytics roboflow
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")

## 3. Download the dataset from Roboflow

In [ ]:
import os, yaml
from getpass import getpass
from roboflow import Roboflow

try:
    from google.colab import userdata
    api_key = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    api_key = None
if not api_key:
    api_key = getpass("Roboflow API key: ")

rf = Roboflow(api_key=api_key)
dataset = (rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
             .version(ROBOFLOW_VERSION).download("yolov8"))

# Rewrite data.yaml with absolute paths (Roboflow exports often have paths Ultralytics cannot find).
data_yaml = os.path.join(dataset.location, "data.yaml")
with open(data_yaml) as f:
    data = yaml.safe_load(f)
data["path"] = dataset.location
data["train"] = "train/images"
if os.path.isdir(os.path.join(dataset.location, "valid", "images")):
    data["val"] = "valid/images"
else:
    print("WARNING: no validation split; validating on the training images. Add a split in Roboflow.")
    data["val"] = "train/images"
if os.path.isdir(os.path.join(dataset.location, "test", "images")):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(data_yaml, "w") as f:
    yaml.safe_dump(data, f)

names = data["names"]
dataset_classes = list(names.values()) if isinstance(names, dict) else list(names)
print("Dataset classes:", dataset_classes)
for split in ("train", "valid", "test"):
    folder = os.path.join(dataset.location, split, "images")
    if os.path.isdir(folder):
        print(f"{split}: {len(os.listdir(folder))} images")

missing = [c for c in CLASSES if c not in dataset_classes]
assert not missing, f"CLASSES {missing} are not in this dataset. Fix the spelling in the parameters cell."

## 4. Train

In [ ]:
from pathlib import Path
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
model.train(data=data_yaml, epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH,
            project="runs", name="stockvision", exist_ok=True)
best = Path(model.trainer.save_dir) / "weights" / "best.pt"
print("Best weights:", best)

## 5. Validate: accuracy for the pilot classes

Copy these numbers into the pilot scope document and the presentation.

In [ ]:
best_model = YOLO(str(best))
metrics = best_model.val(data=data_yaml, imgsz=IMAGE_SIZE)
print(f"\nAll classes: mAP50 {metrics.box.map50:.3f}   mAP50-95 {metrics.box.map:.3f}\n")
print(f"{'class':<20}{'precision':>10}{'recall':>10}{'mAP50':>10}")
for i, class_index in enumerate(metrics.ap_class_index):
    name = best_model.names[int(class_index)]
    if name in CLASSES:
        p, r, map50, _ = metrics.box.class_result(i)
        print(f"{name:<20}{p:>10.3f}{r:>10.3f}{map50:>10.3f}")

## 6. Download the weights and sample images

Put `best.pt` in the repo's `weights/` folder. Unzip `samples.zip` into `data/samples/`.

In [ ]:
import shutil, zipfile
shutil.copy(best, "best.pt")

sample_dir = next((os.path.join(dataset.location, s, "images") for s in ("test", "valid", "train")
                   if os.path.isdir(os.path.join(dataset.location, s, "images"))), None)
with zipfile.ZipFile("samples.zip", "w") as z:
    for name in sorted(os.listdir(sample_dir))[:8]:
        z.write(os.path.join(sample_dir, name), name)

try:
    from google.colab import files
    files.download("best.pt")
    files.download("samples.zip")
except ImportError:
    print("Not on Colab: best.pt and samples.zip are in the current folder.")